# Edge-Ready ECG Classification using CRISP-DM

## Setup

In [39]:

from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb

# Paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MITDB = PROJECT_ROOT / "data" / "mit-bih-arrhythmia-database-1.0.0"
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

assert MITDB.is_dir(), f"database folder not found: {MITDB}"
assert (MITDB / "100.dat").exists(), f"no records inside {MITDB}"

# Record split, DS1 is development,  DS2 is final test set
DS1 = ["101", "106", "108", "109", "112", "114", "115", "116", "118", "119",
       "122", "124", "201", "203", "205", "207", "208", "209", "215", "220",
       "223", "230"]

DS2 = ["100", "103", "105", "111", "113", "117", "121", "123", "200", "202",
       "210", "212", "213", "214", "219", "221", "222", "228", "231", "232",
       "233", "234"]

# Removed because nearly all of the beats are paced "/"
EXCLUDED_RECORDS = ["102", "104", "107", "217"]

# MIT-BIH is 48 records from 47 subjects: records 201 and 202 come from the
# same subject. Both are in the split, so 44 records come from 43 subjects
N_SUBJECTS = 43

# Annotation symbols
# Symbols that mark an actual heartbeat
BEAT_SYMBOLS = {"N", "L", "R", "e", "j", "A", "a", "J", "S", "V", "E", "F",
                "/", "f", "Q"}

# Symbols that mark an event in the signal but carry no heartbeat
NON_BEAT = {"+", "~", "|", "!", "[", "]", "x", '"', "s", "T", "*", "D", "=",
            "p", "^", "t", "u", "?", "(", ")", "@"}

# AAMI EC57 grouping
AAMI_MAP = {
    "N": "N", # Normal beat
    "L": "N", # Left bundle branch block beat
    "R": "N", # Right bundle branch block beat
    "e": "N", # Atrial escape beat
    "j": "N", # Nodal escape beat
    "A": "S", # Atrial premature contraction
    "a": "S", # Aberrated atrial premature beat
    "J": "S", # Nodal premature beat
    "S": "S", # Supraventricular premature or ectopic beat
    "V": "V", # Premature ventricular contraction
    "E": "V", # Ventricular escape beat
    "F": "F", # Fusion of ventricular and normal beat
    "/": "Q", # Paced beat
    "f": "Q", # Fusion of paced and normal beat
    "Q": "Q", # Unclassifiable beat
}

CLASSES = ["N", "S", "V", "F"]
LEAD = "MLII"

print(f"Database: {MITDB}")
print(f"Records:  {len(DS1)} DS1 + {len(DS2)} DS2 = {len(DS1) + len(DS2)}")

Database: /Users/p4ffos/Projects/Study/LTU Projects (MSc)/Data_Mining/Group19_ESG_Project/data/mit-bih-arrhythmia-database-1.0.0
Records:  22 DS1 + 22 DS2 = 44


## 2.1 Dataset description

In [31]:
def collect_metadata(records, split_name):
    rows = []
    for record_id in records:
        header = wfdb.rdheader(str(MITDB / record_id))
        leads = [name.strip() for name in header.sig_name]
        rows.append(
            {
                "record": record_id,
                "split": split_name,
                "fs_hz": header.fs,
                "n_channels": header.n_sig,
                "n_samples": header.sig_len,
                "duration_min": round(header.sig_len / header.fs / 60, 2),
                "channel_0": leads[0],
                "channel_1": leads[1] if len(leads) > 1 else "",
                "units": "/".join(header.units),
            }
        )
    return rows

metadata = pd.DataFrame(collect_metadata(DS1, "DS1") + collect_metadata(DS2, "DS2"))
display(metadata)
metadata.to_csv(RESULTS / "record_metadata.csv", index=False)

,record,split,fs_hz,n_channels,n_samples,duration_min,channel_0,channel_1,units
0,101,DS1,360,2,650000,30.09,MLII,V1,mV/mV
1,106,DS1,360,2,650000,30.09,MLII,V1,mV/mV
2,108,DS1,360,2,650000,30.09,MLII,V1,mV/mV
3,109,DS1,360,2,650000,30.09,MLII,V1,mV/mV
4,112,DS1,360,2,650000,30.09,MLII,V1,mV/mV
5,114,DS1,360,2,650000,30.09,V5,MLII,mV/mV
6,115,DS1,360,2,650000,30.09,MLII,V1,mV/mV
7,116,DS1,360,2,650000,30.09,MLII,V1,mV/mV
8,118,DS1,360,2,650000,30.09,MLII,V1,mV/mV
9,119,DS1,360,2,650000,30.09,MLII,V1,mV/mV


In [32]:
print("Database: MIT-BIH Arrhythmia Database v1.0.0 (PhysioNet)")
print("Number of records: " + str(len(metadata)) + " of 48")
print("Records dropped: " + ", ".join(EXCLUDED_RECORDS))
print("Number of subjects: " + str(N_SUBJECTS) + " (records 201 and 202 share a subject)")
print("Number of channels: " + str(metadata["n_channels"].unique()))
print("Sampling frequency: " + str(metadata["fs_hz"].unique()) + " Hz")

total_minutes = metadata["duration_min"].sum()
print(f"Recording duration: {total_minutes:.0f} min in total")
print("Annotation type: beat-by-beat")


Database: MIT-BIH Arrhythmia Database v1.0.0 (PhysioNet)
Number of records: 44 of 48
Records dropped: 102, 104, 107, 217
Number of subjects: 43 (records 201 and 202 share a subject)
Number of channels: [2]
Sampling frequency: [360] Hz
Recording duration: 1324 min in total
Annotation type: beat-by-beat


In [ ]:
# ECG leads available
print("ECG leads available:")
lead_pairs = metadata.groupby(["channel_0", "channel_1"]).size()
print(lead_pairs)

# Which records have MLII, and in which channel position
in_channel_0 = []
in_channel_1 = []
missing_lead = []

for i in range(len(metadata)):
    row = metadata.iloc[i]
    if row["channel_0"] == LEAD:
        in_channel_0.append(row["record"])
    elif row["channel_1"] == LEAD:
        in_channel_1.append(row["record"])
    else:
        missing_lead.append(row["record"])

print(f"\n{LEAD} in channel 0: {len(in_channel_0)} records")
print(f"{LEAD} in channel 1: {len(in_channel_1)} records {in_channel_1}")
print(f"{LEAD} not present: {len(missing_lead)} records {missing_lead}")

ECG leeds available:
channel_0  channel_1
MLII       V1           38
           V2            2
           V4            1
           V5            2
V5         MLII          1
dtype: int64

MLII in channel 0: 43 records
MLII in channel 1: 1 records ['114']
MLII not present: 0 records []


## 2.2 Target classes

In [34]:
def collect_counts(records, split_name):
    rows = []
    for record_id in records:
        annotation = wfdb.rdann(str(MITDB / record_id), "atr")
        counts = Counter(annotation.symbol)
        for symbol, count in counts.items():
            if symbol not in BEAT_SYMBOLS and symbol not in NON_BEAT:
                raise ValueError(f"Unknown symbol {symbol!r} in record {record_id}")
            rows.append(
                {
                    "split": split_name,
                    "record": record_id,
                    "symbol": symbol,
                    "count": count,
                    "is_beat": symbol in BEAT_SYMBOLS,
                    "aami_class": AAMI_MAP.get(symbol, "-"),
                }
            )
    return rows

df = pd.DataFrame(collect_counts(DS1, "DS1") + collect_counts(DS2, "DS2"))

display(df.head())
print("Rows:", len(df))
print("Records:", df["record"].nunique())
print("Total annotations:", df["count"].sum())
print("Beats:", df.loc[df["is_beat"], "count"].sum())
print("Markers:", df.loc[~df["is_beat"], "count"].sum())
print("Distinct symbols:", sorted(df["symbol"].unique()))

,split,record,symbol,count,is_beat,aami_class
0,DS1,101,+,1,False,-
1,DS1,101,N,1860,True,N
2,DS1,101,~,4,False,-
3,DS1,101,|,4,False,-
4,DS1,101,Q,2,True,Q


Rows: 260
Records: 44
Total annotations: 103724
Beats: 100733
Markers: 2991
Distinct symbols: ['!', '"', '+', 'A', 'E', 'F', 'J', 'L', 'N', 'Q', 'R', 'S', 'V', '[', ']', 'a', 'e', 'j', 'x', '|', '~']


In [35]:
def total_count(frame, split_name):
    return int(frame.loc[frame["split"] == split_name, "count"].sum())

# Table A: every raw annotation symbol
rows = []
for symbol in sorted(df["symbol"].unique()):
    subset = df[df["symbol"] == symbol]
    ds1 = total_count(subset, "DS1")
    ds2 = total_count(subset, "DS2")
    rows.append(
        {
            "symbol": symbol,
            "is_beat": bool(subset["is_beat"].iloc[0]),
            "aami_class": subset["aami_class"].iloc[0],
            "DS1": ds1,
            "DS2": ds2,
            "total": ds1 + ds2,
        }
    )

symbol_table = pd.DataFrame(rows)
symbol_table = symbol_table.sort_values(["is_beat", "total"], ascending=[False, False])

display(symbol_table)
symbol_table.to_csv(RESULTS / "symbol_counts.csv", index=False)

,symbol,is_beat,aami_class,DS1,DS2,total
8,N,True,N,38102,36444,74546
7,L,True,N,3949,4126,8075
10,R,True,N,3783,3476,7259
12,V,True,V,3683,3220,6903
3,A,True,S,810,1736,2546
5,F,True,F,415,388,803
17,j,True,N,16,213,229
15,a,True,S,100,50,150
4,E,True,V,105,1,106
6,J,True,S,32,51,83


In [36]:
# Table B: the four target classes
beats = df[df["is_beat"]]

ds1_kept = 0
ds2_kept = 0
for class_name in CLASSES:
    class_rows = beats[beats["aami_class"] == class_name]
    ds1_kept = ds1_kept + total_count(class_rows, "DS1")
    ds2_kept = ds2_kept + total_count(class_rows, "DS2")

rows = []
for class_name in CLASSES + ["Q"]:
    class_rows = beats[beats["aami_class"] == class_name]
    ds1 = total_count(class_rows, "DS1")
    ds2 = total_count(class_rows, "DS2")
    symbols_present = sorted(class_rows["symbol"].unique())
    rows.append(
        {
            "class": class_name,
            "annotations_included": ", ".join(symbols_present),
            "DS1": ds1,
            "DS2": ds2,
            "total": ds1 + ds2,
            "% DS1": round(100 * ds1 / ds1_kept, 2),
            "% DS2": round(100 * ds2 / ds2_kept, 2),
        }
    )

class_table = pd.DataFrame(rows)

display(class_table)
class_table.to_csv(RESULTS / "class_counts.csv", index=False)

print(f"Beats kept: DS1={ds1_kept}  DS2={ds2_kept}")

,class,annotations_included,DS1,DS2,total,% DS1,% DS2
0,N,"L, N, R, e, j",45866,44259,90125,89.91,89.04
1,S,"A, J, S, a",944,1837,2781,1.85,3.70
2,V,"E, V",3788,3221,7009,7.43,6.48
3,F,F,415,388,803,0.81,0.78
4,Q,Q,8,7,15,0.02,0.01


Beats kept: DS1=51013  DS2=49705


## 2.3 Exploratory Data Analysis

In [ ]:
# TODO

## 2.4 Data Quality

In [52]:
import numpy as np

# 10 s = 3600 samples at 360 Hz. Long enough to hold several beats, short
# enough that a noisy stretch is not diluted by the whole recording
WINDOW_SECONDS = 10


def collect_signal_quality(records, split_name):
    rows = []
    for record_id in records:
        record = wfdb.rdrecord(str(MITDB / record_id))
        lead_names = [name.strip() for name in record.sig_name]

        # Select the lead by NAME
        channel_index = lead_names.index(LEAD)
        signal = record.p_signal[:, channel_index]

        # Drop NaN first, otherwise one bad sample turns every statistic into NaN
        is_valid = np.isfinite(signal)
        signal = signal[is_valid]

        # Split into windows and compare the worst one against the typical one
        window_size = WINDOW_SECONDS * record.fs
        n_windows = len(signal) // window_size
        windows = signal[: n_windows * window_size].reshape(n_windows, window_size)
        window_std = windows.std(axis=1)

        low = float(np.percentile(signal, 0.1))
        high = float(np.percentile(signal, 99.9))

        rows.append(
            {
                "record": record_id,
                "split": split_name,
                "missing_samples": int((~is_valid).sum()),
                "amplitude_min_mV": round(float(signal.min()), 3),
                "amplitude_max_mV": round(float(signal.max()), 3),
                # 99.9th minus 0.1st percentile: the range the signal
                # actually occupies, ignoring isolated spikes.
                "typical_range_mV": round(high - low, 3),
                "amplitude_std_mV": round(float(signal.std()), 3),
                "disturbance_ratio": round(
                    float(window_std.max()) / float(np.median(window_std)), 2
                ),
            }
        )
    return rows


signal_quality = pd.DataFrame(
    collect_signal_quality(DS1, "DS1") + collect_signal_quality(DS2, "DS2")
)

display(signal_quality)
signal_quality.to_csv(RESULTS / "signal_quality.csv", index=False)

,record,split,missing_samples,amplitude_min_mV,amplitude_max_mV,typical_range_mV,amplitude_std_mV,disturbance_ratio
0,101,DS1,0,-3.175,2.420,2.550,0.261,3.73
1,106,DS1,0,-2.065,2.570,3.475,0.368,1.44
2,108,DS1,0,-3.035,2.365,3.970,0.301,4.91
3,109,DS1,0,-3.195,3.130,4.060,0.499,2.09
4,112,DS1,0,-2.635,0.820,1.880,0.224,2.52
5,114,DS1,0,-2.505,3.770,4.040,0.375,3.10
6,115,DS1,0,-2.885,2.410,3.740,0.363,2.35
7,116,DS1,0,-5.120,5.115,8.270,0.664,4.23
8,118,DS1,0,-3.090,2.395,3.345,0.428,2.29
9,119,DS1,0,-2.985,2.500,4.610,0.535,1.43


In [ ]:
# missing data
print(
    "Missing data:",
    int((signal_quality["missing_samples"] > 0).sum()),
    "of",
    len(signal_quality),
)

# differences between recordings
std_min = signal_quality["amplitude_std_mV"].min()
std_max = signal_quality["amplitude_std_mV"].max()
print(
    f"\nAmplitude: {std_min:.3f} to {std_max:.3f} mV "
    f"({std_max / std_min:.1f}x spread between records)"
)

# extreme amplitudes: min/max far outside typical_range means isolated spikes
print("\nWidest signals:")
display(signal_quality.nlargest(5, "typical_range_mV"))

# possible noisy regions
print("\nMost disturbed records:")
display(signal_quality.nlargest(5, "disturbance_ratio"))
print("Calmest records:")
display(signal_quality.nsmallest(5, "disturbance_ratio"))

# Differences between ECG channels
for record_id in ["100", "103", "105", "114"]:
    record = wfdb.rdrecord(str(MITDB / record_id))
    names = [name.strip() for name in record.sig_name]
    print(
        f"{record_id}: {names[0]} std {record.p_signal[:, 0].std():.3f}, "
        f"{names[1]} std {record.p_signal[:, 1].std():.3f}"
    )

# Corrupted segments
record = wfdb.rdrecord(str(MITDB / "116"))
x = record.p_signal[:, 0]
print("\nRecord 116 samples at -5.11 or below:", (x <= -5.11).sum())
print("Record 116 samples at 5.11 or above:", (x >= 5.11).sum())

Missing data: 0 of 44

Amplitude: 0.165 to 0.671 mV (4.1x spread between records)

Widest signals:


,record,split,missing_samples,amplitude_min_mV,amplitude_max_mV,typical_range_mV,amplitude_std_mV,disturbance_ratio
7,116,DS1,0,-5.120,5.115,8.270,0.664,4.23
13,203,DS1,0,-4.305,3.640,4.835,0.495,2.52
34,213,DS2,0,-2.530,3.285,4.700,0.671,1.38
24,105,DS2,0,-3.715,3.000,4.650,0.405,3.43
9,119,DS1,0,-2.985,2.500,4.610,0.535,1.43



Most disturbed records:


,record,split,missing_samples,amplitude_min_mV,amplitude_max_mV,typical_range_mV,amplitude_std_mV,disturbance_ratio
28,121,DS2,0,-4.450,1.400,3.38,0.301,5.86
2,108,DS1,0,-3.035,2.365,3.97,0.301,4.91
25,111,DS2,0,-2.820,2.375,2.45,0.257,4.88
7,116,DS1,0,-5.120,5.115,8.27,0.664,4.23
15,207,DS1,0,-3.955,2.285,3.47,0.354,3.78


Calmest records:


,record,split,missing_samples,amplitude_min_mV,amplitude_max_mV,typical_range_mV,amplitude_std_mV,disturbance_ratio
18,215,DS1,0,-1.380,2.500,2.855,0.282,1.24
37,221,DS2,0,-1.390,2.230,2.495,0.301,1.30
35,214,DS2,0,-2.730,2.800,4.300,0.470,1.33
40,231,DS2,0,-1.355,2.050,2.725,0.272,1.33
17,209,DS1,0,-1.455,1.885,2.340,0.268,1.34


100: MLII std 0.193, V5 std 0.148
103: MLII std 0.322, V2 std 0.687
105: MLII std 0.405, V1 std 0.279
114: V5 std 0.171, MLII std 0.375

Record 116 samples at -5.11 or below: 666
Record 116 samples at  5.11 or above: 265
